# Stochastic Calculus from Zero · Episode 1: Why ordinary calculus breaks

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> Educational material only, not financial advice.

1. Setup · 2. The school rule on a smooth path · 3. One random path breaks it · 4. Where the gap lives · 5. Smooth vs rough · 6. Every path · 7. History · 8. This episode's question · 9. Exercises · 10. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 1,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)

## 2. The school rule on a smooth path

For a smooth path $x(t)$, the chain rule says $d(x^2) = 2x\,dx$. Cut a year into $n$ steps, add up $2x\,\Delta x$ along the path (taking $x$ at the start of each step), and you should land on $x_T^2$.

Our smooth path: $x(t) = t + 0.3\sin(2\pi t)$, which wiggles but has no corners. It ends at $x_1 = 1$, so $x_1^2 = 1$. The chain-rule sum gets there, and the sum of squared steps, the part the rule throws away, shrinks to nothing as the steps shrink.

In [ ]:
smooth = lambda t: t + 0.3 * np.sin(2 * np.pi * t)
rows = []
for n in (10, 100, 1_000, 10_000):
    x = smooth(np.linspace(0, 1, n + 1))
    dx = np.diff(x)
    rows.append({"steps": n, "chain_rule_sum": float(2 * (x[:-1] * dx).sum()), "squares": float((dx ** 2).sum()),
                 "x_T_sq": float(x[-1] ** 2)})
    assert abs(rows[-1]["chain_rule_sum"] + rows[-1]["squares"] - rows[-1]["x_T_sq"]) < 1e-12
assert rows[-1]["squares"] < 1e-3 and abs(rows[-1]["chain_rule_sum"] - 1) < 1e-3
out["smooth"] = {"rows": rows, "squares_10000": rows[-1]["squares"]}
pd.DataFrame(rows)

## 3. One random path breaks it

Now a Brownian path (Series 3, Episode 11) over one year with 1,000 steps: each step is a normal draw with variance $1/1000$. Same recipe: add up $2W\Delta W$ with $W$ at the start of each step. The school rule says the total should be $W_T^2$. It isn't.

In [ ]:
rng = np.random.default_rng(seed)
t, W, dW = brownian_paths(1, 1_000, 1.0, rng)
w, dw = W[0], dW[0]
one = {"W_T": float(w[-1]), "W_T_sq": float(w[-1] ** 2), "chain_rule_sum": float(chain_rule_sum(w, dw)),
       "squares": float(sum_of_squares(dw))}
one["gap"] = one["W_T_sq"] - one["chain_rule_sum"]
assert abs(one["gap"] - one["squares"]) < 1e-12        # the gap is exactly the sum of squared steps
assert one["chain_rule_sum"] < 0 < one["W_T"]           # the narration says "minus" and "ends at" a positive level
every = 2   # every second point is plenty for the chart
out["one"] = one | {"steps": 1_000, "t": t[::every].round(4).tolist(), "path": [w[::every].round(4).tolist()],
                    "ticks": [{"x": x, "label": lab} for x, lab in ((0, "0"), (0.25, "3m"), (0.5, "6m"), (0.75, "9m"), (1, "1y"))],
                    "table": [{"what": "W at the end, squared", "value": one["W_T_sq"]},
                              {"what": "Chain rule: add up 2W·ΔW", "value": one["chain_rule_sum"]},
                              {"what": "Gap", "value": one["gap"]}]}
plt.figure(figsize=(8, 3))
plt.plot(t, w, lw=0.8)
plt.title(f"One Brownian path: W_T = {w[-1]:.3f}, W_T^2 = {w[-1] ** 2:.3f}, chain-rule sum = {one['chain_rule_sum']:.3f}")
plt.show()
one

## 4. Where the gap lives

For any two numbers, $b^2 - a^2 = 2a(b - a) + (b - a)^2$. Apply it to every step and add up: the squares telescope, so on **any** path

$$W_T^2 = \sum 2W\,\Delta W + \sum (\Delta W)^2.$$

This is algebra, not probability. The school rule keeps the first sum and drops the second, which is fine for a smooth path (Section 2: the squares vanish) and wrong for a Brownian one.

In [ ]:
lhs = np.diff(w ** 2)                       # b^2 - a^2, step by step
rhs = 2 * w[:-1] * dw + dw ** 2              # 2a(b - a) + (b - a)^2
out["identity_max_error"] = float(np.abs(lhs - rhs).max())
assert out["identity_max_error"] < 1e-12
out["identity_max_error"]

## 5. Smooth vs rough

Why do the squares vanish on one path and not the other? On a smooth path each step is about $\Delta t$, so each square is about $(\Delta t)^2$: $n$ of them add up to about $1/n$. On a Brownian path each step is about $\sqrt{\Delta t}$, so each square is about $\Delta t$, and $n$ of them add up to the whole year.

Below: one smooth path and one 10,000-step Brownian path, each measured with coarser and finer steps.

In [ ]:
_, Wf, _ = brownian_paths(1, 10_000, 1.0, rng)
fine = Wf[0]
compare = []
for n in (10, 100, 1_000, 10_000):
    stride = 10_000 // n
    compare.append({"steps": n, "smooth": float((np.diff(smooth(np.linspace(0, 1, n + 1))) ** 2).sum()),
                    "random": float((np.diff(fine[::stride]) ** 2).sum())})
assert compare[-1]["smooth"] < 1e-3 and abs(compare[-1]["random"] - 1) < 0.05
out["compare"] = {"rows": compare, "smooth_10000": compare[-1]["smooth"], "random_10000": compare[-1]["random"]}
pd.DataFrame(compare)

## 6. Every path

One path could be luck. 10,000 paths of 1,000 steps: on average $W_T^2$ is about 1 (its variance), the chain-rule sum is about 0, and the sum of squared steps is about 1 on **every** path, not just on average. Its spread from path to path is $\sqrt{2/n}$, which goes to zero as the steps shrink: in the limit the squares add up to exactly the time elapsed. Part 2 of the series names it *quadratic variation* and builds on it.

In [ ]:
M = 10_000
_, Wm, dWm = brownian_paths(M, 1_000, 1.0, rng)
chain_m, sq_m = chain_rule_sum(Wm, dWm), sum_of_squares(dWm)
assert np.abs(Wm[:, -1] ** 2 - chain_m - sq_m).max() < 1e-9
paths = {"paths": M, "mean_W_T_sq": float((Wm[:, -1] ** 2).mean()), "mean_chain_rule_sum": float(chain_m.mean()),
         "mean_squares": float(sq_m.mean()), "sd_squares": float(sq_m.std())}
assert abs(paths["mean_chain_rule_sum"]) < 0.05 and abs(paths["mean_squares"] - 1) < 0.01   # "about zero", "about one"
spread = []
for n in (10, 100, 1_000, 10_000):
    _, _, inc = brownian_paths(1_000, n, 1.0, rng)
    spread.append({"steps": n, "sd": float(sum_of_squares(inc).std()), "theory": math.sqrt(2 / n)})
    assert abs(spread[-1]["sd"] / spread[-1]["theory"] - 1) < 0.15
del Wm, dWm
out["paths"] = paths | {"spread": spread, "sd_10": spread[0]["sd"], "sd_10000": spread[-1]["sd"],
                        "table": [{"what": "W at the end, squared", "value": paths["mean_W_T_sq"]},
                                  {"what": "Chain rule: add up 2W·ΔW", "value": paths["mean_chain_rule_sum"]},
                                  {"what": "Sum of squared steps", "value": paths["mean_squares"]}]}
paths, pd.DataFrame(spread)

## 7. History

- **1900.** Louis Bachelier models prices on the Paris Bourse as random in his thesis *Théorie de la spéculation* (Annales scientifiques de l'École Normale Supérieure).
- **1923.** Norbert Wiener puts Brownian motion on a rigorous footing in *Differential-Space* (Journal of Mathematics and Physics, vol. 2).
- **1940.** Wolfgang (Vincent) Doeblin, a soldier in the French army, deposits a sealed envelope, *Sur l'équation de Kolmogoroff*, with the Académie des sciences on 26 February 1940. He died in June 1940; the envelope was opened on 18 May 2000 and found to contain a change-of-variables formula for diffusions.
- **1944.** Kiyosi Itô publishes *Stochastic integral* (Proceedings of the Imperial Academy, Tokyo, vol. 20), the start of Itô calculus.

The formula Part 2 builds (Episode 15) is now often called the Itô-Doeblin formula.

In [ ]:
out["history"] = {"bachelier": 1900, "wiener": 1923, "doeblin_deposit": 1940, "ito": 1944, "doeblin_opened": 2000,
                  "doeblin_wait": 2000 - 1940}
out["history"]

## 8. This episode's question

Run the path for two years instead of one. Where does the sum of squared steps settle? Each step's square averages $\Delta t$, so the total averages the time elapsed: 2.

In [ ]:
_, _, inc2 = brownian_paths(1_000, 2_000, 2.0, rng)
q = sum_of_squares(inc2)
out["question"] = {"years": 2, "mean_squares": float(q.mean()), "sd_squares": float(q.std())}
assert abs(out["question"]["mean_squares"] - 2) < 0.02
out["question"]

## 9. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** A path takes two steps: $W_0 = 0$, $W_1 = a$, $W_2 = b$. Check the identity $W_2^2 = \sum 2W\Delta W + \sum (\Delta W)^2$ by expanding both sides.

<details><summary>Answer</summary>

Right side: $2\cdot 0\cdot a + 2a(b - a) + a^2 + (b - a)^2 = 2ab - 2a^2 + a^2 + b^2 - 2ab + a^2 = b^2$. ✓

</details>

**Exercise 2 (By hand).** For the smooth path $x(t) = t^2$ on $[0, 1]$ with $n$ equal steps, compute $\sum (\Delta x)^2$. What happens as $n$ grows?

<details><summary>Answer</summary>

$\Delta x_i = \frac{(i+1)^2 - i^2}{n^2} = \frac{2i + 1}{n^2}$, so $\sum_{i=0}^{n-1} (\Delta x_i)^2 = \frac{1}{n^4}\sum (2i+1)^2 = \frac{1}{n^4}\cdot\frac{n(2n-1)(2n+1)}{3} = \frac{4n^2 - 1}{3n^3} \approx \frac{4}{3n} \to 0$.

</details>

In [ ]:
#@title Solution 2 (double-click to show the code) { display-mode: "form" }
for n in (10, 100, 1000):
    dx = np.diff(np.linspace(0, 1, n + 1) ** 2)
    assert abs((dx ** 2).sum() - (4 * n ** 2 - 1) / (3 * n ** 3)) < 1e-12
print("sum of squared steps of t^2 = (4n^2 - 1) / (3n^3): checked for n = 10, 100, 1000")

**Exercise 3 (By hand).** A Brownian path on $[0, T]$ is cut into $n$ steps. What is the average of $\sum (\Delta W)^2$? Its standard deviation? (Hint: for a normal $Z$ with variance $v$, $E[Z^2] = v$ and $\mathrm{Var}(Z^2) = 2v^2$.)

<details><summary>Answer</summary>

Each step has variance $T/n$, so the average is $n \cdot T/n = T$ for every $n$. The steps are independent, so the variances add: $n \cdot 2(T/n)^2 = 2T^2/n$, and the standard deviation is $T\sqrt{2/n}$, which goes to 0.

</details>

**Exercise 4 (In code).** Plot the path-to-path spread of $\sum (\Delta W)^2$ against $n$ on log-log axes (use Section 6's `spread`). Fit a straight line: what is its slope, and why?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
n = np.array([r["steps"] for r in spread]); sd = np.array([r["sd"] for r in spread])
slope = np.polyfit(np.log(n), np.log(sd), 1)[0]
plt.figure(figsize=(5, 3)); plt.loglog(n, sd, "o-"); plt.xlabel("steps"); plt.ylabel("spread"); plt.show()
assert -0.55 < slope < -0.45     # spread = sqrt(2/n): slope -1/2
print(f"slope {slope:.3f}: the spread falls like 1/sqrt(n)")

**Exercise 5 (In code).** Redo the chain-rule sum with $W$ taken at the **end** of each step instead of the start: $\sum 2W_{\text{end}}\Delta W$. What does it average over many paths, and how far is it from $W_T^2$? (This is why Part 2's integral insists on the start of each step.)

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
_, Wx, dWx = brownian_paths(5_000, 1_000, 1.0, np.random.default_rng(seed + 1))
right = 2 * (Wx[:, 1:] * dWx).sum(axis=1)
# W_end = W_start + dW, so the right-point sum = left-point sum + 2 x squares = W_T^2 + squares
assert np.abs(right - (Wx[:, -1] ** 2 + sum_of_squares(dWx))).max() < 1e-9
assert abs(right.mean() - 2) < 0.1
print(f"right-point sum averages {right.mean():.3f} (= 2T): it overshoots W_T^2 by the sum of squared steps")

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")